In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
#from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

Matplotlib is building the font cache; this may take a moment.


In [2]:
df = pd.read_csv(r"C:\Users\AiK\SmartShelf AI\Dataset\messy.csv")
df.head()

,Product_ID,Store_ID,Category,Supplier,Region,Store_Size,Snapshot_Date,Season,Current_Stock,Daily_Sales,...,Weather_Impact,Is_Perishable,Stock_Coverage_Days,Price_per_Demand,Supplier_Lead_Reliability,Store_Age_Years,Last_Audit_Score,Competitor_Price_Index,Random_Noise_A,Out_of_Stock
0,P00468,S046,Meat,NaN,South,Small,2025-07-22,Winter,12,3.05,...,Low,1,3.81,1.073,1.167,23,83.8,4.35,41.24,0
1,P00923,S052,Meat,Supplier_B2,East,Small,2025-08-17,Winter,38,1.56,...,Medium,1,22.89,3.189,0.929,10,65.5,6.04,50.37,0
2,P00069,S011,Dairy,Supplier_E2,Central,Medium,2025-06-08,Winter,15,4.05,...,Medium,1,3.61,1.031,0.724,30,74.9,4.88,62.16,0
3,P01418,S024,Bakery,Supplier_O1,North,Large,2025-06-21,Spring,55,9.27,...,Low,1,5.87,0.427,1.148,17,92.8,5.18,38.85,0
4,P02249,S045,Produce,Supplier_F2,Central,Medium,2025-06-01,Winter,39,3.25,...,Medium,1,11.64,0.741,1.242,2,99.4,3.03,34.46,0


In [3]:
columns = df.shape[1]
rows = df.shape[0]
print(f"Rows : {rows} || Columns : {columns}")

Rows : 150000 || Columns : 29


In [4]:
df.describe()

,Current_Stock,Daily_Sales,Reorder_Level,Lead_Time_Days,Unit_Price,Discount_Percent,Shelf_Capacity,Days_Since_Last_Restock,Customer_Demand_Index,Is_Perishable,Stock_Coverage_Days,Price_per_Demand,Supplier_Lead_Reliability,Store_Age_Years,Last_Audit_Score,Competitor_Price_Index,Random_Noise_A,Out_of_Stock
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,146915.000000,150000.000000,149066.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,145411.000000,147709.000000,150000.000000,150000.000000
mean,23.766393,4.475632,25.148173,3.136780,4.129827,3.078787,49.803120,3.084721,65.856179,0.517840,8.006417,1.629303,0.982407,17.493073,80.044481,4.338101,50.017858,0.181053
std,25.610574,6.036253,39.602349,2.579415,3.232378,6.937619,39.222833,3.589918,14.992095,0.499683,7.842939,2.043215,0.155150,9.813228,11.538489,3.446936,15.011653,0.385064
min,0.000000,0.100000,2.000000,1.000000,0.460000,0.000000,9.000000,0.000000,21.200000,0.000000,0.000000,0.013000,0.710000,1.000000,60.000000,0.390000,-22.910000,0.000000
25%,8.000000,1.350000,5.000000,1.000000,2.040000,0.000000,27.000000,1.000000,55.200000,0.000000,2.760000,0.467000,0.872000,9.000000,70.100000,2.110000,39.890000,0.000000
50%,17.000000,2.670000,12.000000,2.000000,3.200000,0.000000,39.000000,2.000000,63.400000,1.000000,5.990000,0.981000,0.977000,18.000000,80.100000,3.340000,50.020000,0.000000
75%,30.000000,5.260000,27.000000,4.000000,5.130000,0.000000,58.000000,4.000000,74.000000,1.000000,10.670000,1.991000,1.102000,26.000000,90.050000,5.400000,60.150000,0.000000
max,378.000000,327.330000,300.000000,18.000000,31.090000,40.000000,455.000000,45.000000,100.000000,1.000000,110.000000,32.580000,1.262000,34.000000,100.000000,38.060000,112.060000,1.000000


OUTLIERS DETECTION

In [5]:
df = df.drop(["Product_ID","Store_ID"], axis=1)
df.columns

Index(['Category', 'Supplier', 'Region', 'Store_Size', 'Snapshot_Date',
       'Season', 'Current_Stock', 'Daily_Sales', 'Reorder_Level',
       'Lead_Time_Days', 'Unit_Price', 'Discount_Percent', 'Shelf_Capacity',
       'Promotion', 'Holiday_Week', 'Days_Since_Last_Restock',
       'Customer_Demand_Index', 'Weather_Impact', 'Is_Perishable',
       'Stock_Coverage_Days', 'Price_per_Demand', 'Supplier_Lead_Reliability',
       'Store_Age_Years', 'Last_Audit_Score', 'Competitor_Price_Index',
       'Random_Noise_A', 'Out_of_Stock'],
      dtype='str')

In [13]:
outlier_summary = []

for col in df.select_dtypes(include=np.number).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary.append({
            "Feature": col,
            "Outliers": outliers,
            "Percentage": round((outliers / len(df)) * 100, 2)
        })
        
outlier_df = pd.DataFrame(outlier_summary)

print(outlier_df)

                      Feature  Outliers  Percentage
0               Current_Stock      9019        6.01
1                 Daily_Sales     12095        8.06
2               Reorder_Level     14236        9.49
3              Lead_Time_Days      6936        4.62
4                  Unit_Price      9121        6.08
5            Discount_Percent     32334       21.56
6              Shelf_Capacity     10944        7.30
7     Days_Since_Last_Restock     10977        7.32
8       Customer_Demand_Index        27        0.02
9               Is_Perishable         0        0.00
10        Stock_Coverage_Days      7650        5.10
11           Price_per_Demand     11239        7.49
12  Supplier_Lead_Reliability         0        0.00
13            Store_Age_Years         0        0.00
14           Last_Audit_Score         0        0.00
15     Competitor_Price_Index      8990        5.99
16             Random_Noise_A      1000        0.67
17               Out_of_Stock     27158       18.11


In [15]:
for col in df.select_dtypes(include=np.number).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])
    

In [16]:
for col in df.select_dtypes(include=np.number).columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    print(col, ((df[col] < lower) | (df[col] > upper)).sum())

Current_Stock 0
Daily_Sales 0
Reorder_Level 0
Lead_Time_Days 0
Unit_Price 0
Discount_Percent 0
Shelf_Capacity 0
Days_Since_Last_Restock 0
Customer_Demand_Index 0
Is_Perishable 0
Stock_Coverage_Days 0
Price_per_Demand 0
Supplier_Lead_Reliability 0
Store_Age_Years 0
Last_Audit_Score 0
Competitor_Price_Index 0
Random_Noise_A 0
Out_of_Stock 0


In [17]:
df["Out_of_Stock"].value_counts()

Out_of_Stock
0.0    150000
Name: count, dtype: int64

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 27 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Category                   150000 non-null  str    
 1   Supplier                   148222 non-null  str    
 2   Region                     150000 non-null  str    
 3   Store_Size                 150000 non-null  str    
 4   Snapshot_Date              150000 non-null  str    
 5   Season                     150000 non-null  str    
 6   Current_Stock              150000 non-null  float64
 7   Daily_Sales                150000 non-null  float64
 8   Reorder_Level              150000 non-null  float64
 9   Lead_Time_Days             150000 non-null  float64
 10  Unit_Price                 150000 non-null  float64
 11  Discount_Percent           146915 non-null  float64
 12  Shelf_Capacity             150000 non-null  float64
 13  Promotion                  150000 non-nu

In [19]:
mvcols = []
fcols = []

for col in df.columns:
    if df[col].isnull().sum() == 0:
        fcols.append(col)      # No missing values
    else:
        mvcols.append(col)     # Has missing values

mv = pd.DataFrame({
    "Column": mvcols,
    "Missing Values": [df[col].isnull().sum() for col in mvcols],
    "Percentage": [round(df[col].isnull().mean() * 100, 2) for col in mvcols]
})

print(mv)

                    Column  Missing Values  Percentage
0                 Supplier            1778        1.19
1         Discount_Percent            3085        2.06
2  Days_Since_Last_Restock             934        0.62
3           Weather_Impact            3759        2.51
4         Last_Audit_Score            4589        3.06
5   Competitor_Price_Index            2291        1.53


In [27]:
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include=['object', 'string', 'category']).columns

for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [28]:
df.isnull().sum()

Category                     0
Supplier                     0
Region                       0
Store_Size                   0
Snapshot_Date                0
Season                       0
Current_Stock                0
Daily_Sales                  0
Reorder_Level                0
Lead_Time_Days               0
Unit_Price                   0
Discount_Percent             0
Shelf_Capacity               0
Promotion                    0
Holiday_Week                 0
Days_Since_Last_Restock      0
Customer_Demand_Index        0
Weather_Impact               0
Is_Perishable                0
Stock_Coverage_Days          0
Price_per_Demand             0
Supplier_Lead_Reliability    0
Store_Age_Years              0
Last_Audit_Score             0
Competitor_Price_Index       0
Random_Noise_A               0
Out_of_Stock                 0
dtype: int64

In [30]:
df.to_csv("Cleaned_Data.csv")